# Applied Linear Algebra Assignment
### Author: Kirubel Wubet
### Date: July 2026

---

This notebook evaluates fundamental understanding of linear algebra concepts — specifically **Span**, **Basis**, and **Linear Transformations** — implemented from scratch in raw Python without any external libraries.

All vectors are represented as Python tuples, and all matrices as lists of lists (row-major order, matching standard mathematical convention).

---

## Task 1: The Restricted Robot — Span & Basis

### Problem Setup

An AI agent operates on a 2D grid. Its movement is restricted to linear combinations of exactly two directional vectors:

| Move | Forward (x) | Sideways (y) | Vector Notation |
|------|------------|--------------|-----------------|
| Move A | +2 | +1 | **a = (2, 1)** |
| Move B | +1 | -2 | **b = (1, -2)** |

The robot executes `α` copies of Move A and `β` copies of Move B. Its final position is:
position = α·A + β·B
This is the definition of a **linear combination**. The set of ALL positions reachable this way is called the **span** of {A, B}.

### The Mathematical Question

Given a target coordinate `(x, y)`, we ask: do integer values of `α` and `β` exist such that:
2α +  β = x
α - 2β = y
### Why This Is a Basis Question

Two vectors form a **basis** for ℝ² if they are:
1. **Linearly independent** — neither is a scalar multiple of the other
2. **Spanning** — their combinations cover the entire plane

The determinant of the matrix formed by these two vectors tells us both things at once:
det([[2, 1], [1, -2]]) = (2)(−2) − (1)(1) = −4 − 1 = −5

Since det ≠ 0, the vectors are linearly independent and span all of ℝ². However, the robot is restricted to **integer** multiples, so it only reaches a sublattice — exactly 1 in every |det| = **5** integer points.


In [2]:
# SECTION 1: Core Vector Operations

#scalar multiplication of a vector
def scalar_multiply(scalar, vector):
    return (scalar * vector[0], scalar * vector[1])

#addition of two vectors
def vector_add(v1, v2):
    return (v1[0] + v2[0], v1[1] + v2[1])

#linear combination of two vectors
def linear_combination(alpha, beta, vecA, vecB):
    return vector_add(scalar_multiply(alpha, vecA), scalar_multiply(beta, vecB))

### Step 2: Computing the Determinant

Before we can solve for α and β, we need to confirm the two move vectors are **linearly independent** — meaning they point in genuinely different directions and neither can be made from multiples of the other.

The **determinant** of the 2×2 matrix formed by placing the vectors as columns is the test:
M = [[2,  1],
[1, -2]]
det(M) = (2)(−2) − (1)(1) = −5

**Interpretation:**
- `det ≠ 0` → vectors are linearly independent → they form a basis → every point in ℝ² is reachable with *real* α, β
- `det = 0` → vectors are parallel → they only span a line → most of the plane is permanently unreachable

The absolute value |det| = 5 is also geometrically meaningful: it is the area of the parallelogram formed by the two move vectors. In the context of integer arithmetic, it tells us that only 1 in every 5 integer grid points lies in the robot's reachable sublattice.


In [9]:
# SECTION 2: Determinant and Linear Independence Check

def determinant_2x2(vecA, vecB):
    return (vecA[0] * vecB[1]) - (vecA[1] * vecB[0])


def are_linearly_independent(vecA, vecB):
    return determinant_2x2(vecA, vecB) != 0

### Step 3: Solving the System — Cramer's Rule

We need to find integer α and β satisfying:
2α +  β = x    →    row 1
α − 2β = y    →    row 2

We solve this using **Cramer's Rule** for 2×2 systems. Given `det = −5`:

We solve this using **Cramer's Rule** for 2×2 systems. Given `det = −5`:

α = det([[x, 1], [y, -2]]) / det(M)  =  ((x)(−2) − (1)(y)) / −5  =  (−2x − y) / −5  =  (2x + y) / 5

β = det([[2, x], [1,  y]]) / det(M)  =  ((2)(y)  − (x)(1)) / −5  =  (2y - x)  / −5  =  (x − 2y) / 5

The robot can reach `(x, y)` **if and only if** both `(2x + y)` and `(x − 2y)` are divisible by 5.

In [6]:
# SECTION 3: The Analytical Solver using Cramer's Rule

def solve_cramers_rule(target, vecA, vecB):
    x, y = target

    det_main  = determinant_2x2(vecA, vecB)

    if det_main == 0:
        return {
            'reachable': False,
            'alpha': None,
            'beta': None,
            'det_main': det_main,
            'det_alpha': None,
            'det_beta': None,
            'verification': None,
            'reason': 'Vectors are linearly dependent — they do not span ℝ²'
        }

    # Replace column 1 with target to get det_alpha
    det_alpha = determinant_2x2(target, vecB)   # det([[x, b1],[y, b2]])

    # Replace column 2 with target to get det_beta
    det_beta  = determinant_2x2(vecA, target)   # det([[a1, x],[a2, y]])

    # Check integer divisibility — this is the lattice condition
    alpha_remainder = det_alpha % det_main
    beta_remainder  = det_beta  % det_main

    is_reachable = (alpha_remainder == 0) and (beta_remainder == 0)

    if is_reachable:
        alpha = det_alpha // det_main
        beta  = det_beta  // det_main
        verification = linear_combination(alpha, beta, vecA, vecB)
    else:
        alpha        = None
        beta         = None
        verification = None

    return {
        'reachable':    is_reachable,
        'alpha':        alpha,
        'beta':         beta,
        'det_main':     det_main,
        'det_alpha':    det_alpha,
        'det_beta':     det_beta,
        'verification': verification
    }


def can_reach(target, vecA, vecB):
    """
    Human-readable wrapper around the Cramer's Rule solver.
    Prints a full diagnostic report for a given target coordinate.
    """
    x, y   = target
    result = solve_cramers_rule(target, vecA, vecB)

    print(f"Target: {target}")
    print(f"  det(main)  = {result['det_main']}")
    print(f"  det(alpha) = {result['det_alpha']}  →  α = {result['det_alpha']} / {result['det_main']}", end="")

    if result['reachable']:
        print(f" = {result['alpha']}")
    else:
        print(f" → NOT an integer")

    print(f"  det(beta)  = {result['det_beta']}  →  β = {result['det_beta']} / {result['det_main']}", end="")

    if result['reachable']:
        print(f" = {result['beta']}")
    else:
        print(f" → NOT an integer")

    if result['reachable']:
        print(f"  ✅ REACHABLE  →  {result['alpha']}·A + {result['beta']}·B = {result['verification']}")
    else:
        print(f"  ❌ UNREACHABLE  →  No integer solution exists")

    print()

### Step 4: Brute-Force Validator

Before trusting our analytical solver, we independently verify results using a brute-force loop. This is not the solution — it is a **test harness**. It searches all integer combinations of α and β in a bounded range and checks if any of them produce the target.

If the analytical result and the brute-force result always agree, our solver is correct.

In [7]:
# SECTION 4: Brute-Force Validator (Test Harness Only)

def brute_force_check(target, vecA, vecB, search_range=50):
    """
    Validates reachability by exhaustive search over integer combinations.

    This is a VERIFICATION TOOL, not a solver. It is computationally
    expensive (O(n²) checks) and not scalable, but it serves as an
    independent ground truth to confirm our Cramer's Rule solver.

    Args:
        target       (tuple): The target coordinate
        vecA         (tuple): Move A vector
        vecB         (tuple): Move B vector
        search_range (int):   Searches α, β in [−range, +range]

    Returns:
        tuple: (found: bool, (alpha, beta) or None)
    """
    for alpha in range(-search_range, search_range + 1):
        for beta in range(-search_range, search_range + 1):
            if linear_combination(alpha, beta, vecA, vecB) == target:
                return (True, (alpha, beta))
    return (False, None)

### Step 5: Running All Test Cases

We test four carefully chosen coordinates that cover all cases:

| Target | Expected | Reason |
|--------|----------|--------|
| (3, −1) | ✅ Reachable | α=1, β=1 → 1·A + 1·B = (3,−1) |
| (0, 0) | ✅ Reachable | α=0, β=0 → origin always reachable |
| (1, 0) | ❌ Unreachable | (2·1 + 0) = 2, not divisible by 5 |
| (1, 3) | ✅ Reachable | α=1, β=−1 → 1·A − 1·B = (1,3) |

After running the analytical solver, we cross-check every result with the brute-force validator.

In [10]:
# SECTION 5: Full Test Suite with Cross-Validation


test_cases = [
    (3,  -1),   # Expected: REACHABLE   (α=1, β=1)
    (0,   0),   # Expected: REACHABLE   (α=0, β=0)
    (1,   0),   # Expected: UNREACHABLE
    (1,   3),   # Expected: REACHABLE   (α=1, β=-1)
]

print("=" * 60)
print("       TASK 1 RESULTS: THE RESTRICTED ROBOT")
print("=" * 60)
print(f"Move A = {MOVE_A}   Move B = {MOVE_B}")
print(f"det(M) = {determinant_2x2(MOVE_A, MOVE_B)}")
print("=" * 60)
print()

all_passed = True

for target in test_cases:
    # --- Analytical solver ---
    can_reach(target, MOVE_A, MOVE_B)

    # --- Brute-force cross-check ---
    bf_found, bf_solution = brute_force_check(target, MOVE_A, MOVE_B)
    analytical_result     = solve_cramers_rule(target, MOVE_A, MOVE_B)

    match = (bf_found == analytical_result['reachable'])
    status = "MATCH" if match else "MISMATCH"

    print(f"  Brute-force check: {'Found ' + str(bf_solution) if bf_found else 'Not found'}")
    print(f"  Cross-validation:  {status}")
    print("-" * 60)
    print()

    if not match:
        all_passed = False

print()
print("=" * 60)
if all_passed:
    print("ALL TESTS PASSED — Analytical solver verified correct")
else:
    print("SOME TESTS FAILED — Review solver logic")
print("=" * 60)

       TASK 1 RESULTS: THE RESTRICTED ROBOT
Move A = (2, 1)   Move B = (1, 3)
det(M) = 5

Target: (3, -1)
  det(main)  = 5
  det(alpha) = 10  →  α = 10 / 5 = 2
  det(beta)  = -5  →  β = -5 / 5 = -1
  ✅ REACHABLE  →  2·A + -1·B = (3, -1)

  Brute-force check: Found (2, -1)
  Cross-validation:  ✅ MATCH
------------------------------------------------------------

Target: (0, 0)
  det(main)  = 5
  det(alpha) = 0  →  α = 0 / 5 = 0
  det(beta)  = 0  →  β = 0 / 5 = 0
  ✅ REACHABLE  →  0·A + 0·B = (0, 0)

  Brute-force check: Found (0, 0)
  Cross-validation:  ✅ MATCH
------------------------------------------------------------

Target: (1, 0)
  det(main)  = 5
  det(alpha) = 3  →  α = 3 / 5 → NOT an integer
  det(beta)  = -1  →  β = -1 / 5 → NOT an integer
  ❌ UNREACHABLE  →  No integer solution exists

  Brute-force check: Not found
  Cross-validation:  ✅ MATCH
------------------------------------------------------------

Target: (1, 3)
  det(main)  = 5
  det(alpha) = 0  →  α = 0 / 5 = 0
  de

## Task 1 Summary

### What this task is about
A mathematically rigorous solver that determines whether an AI robot with restricted movement vectors can reach any integer coordinate on a grid.

### Core Concepts Applied

| Concept | Where It Appeared |
|---------|-------------------|
| **Vector as position** | Move A = (2,1), Move B = (1,−2) |
| **Linear combination** | position = α·A + β·B |
| **Span** | The set of ALL reachable positions |
| **Linear independence** | det(M) = −5 ≠ 0 confirms A and B are independent |
| **Basis** | {A, B} forms a basis for ℝ² — every real point is reachable |
| **Integer lattice** | Only 1 in 5 integer points is reachable (due to |det| = 5) |
| **Cramer's Rule** | Analytical formula for α and β without guessing |
| **Determinant** | Computed from scratch; used for independence check AND solving |

### Key Insight
The determinant is doing two jobs simultaneously: it tells us **whether** a solution exists (nonzero = yes) and it gives us **what** the solution is (via Cramer's Rule). This is why the determinant appears at the center of so much linear algebra.